In [5]:
import os
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torch.nn.functional as F
from math import log10
from skimage.metrics import structural_similarity as ssim


class LensingSR_IX_Dataset(Dataset):
    def __init__(self, lr_dir, hr_dir, transform=None):
        self.lr_dir = lr_dir
        self.hr_dir = hr_dir
        self.transform = transform
        self.filenames = sorted([f for f in os.listdir(hr_dir) if f.endswith('.npy')])

    def __len__(self): return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        
        def robust_load(base_path, filename):
            data = np.load(os.path.join(base_path, filename), allow_pickle=True)
            curr = data
            while True:
                if isinstance(curr, np.ndarray) and curr.dtype == object and curr.ndim == 0:
                    curr = curr.item()
                elif isinstance(curr, (list, tuple)) and len(curr) > 0:
                    curr = curr[0]
                elif isinstance(curr, np.ndarray) and curr.dtype == object and curr.ndim > 0:
                    curr = curr[0]
                else: break
            img = np.ascontiguousarray(curr, dtype=np.float32)
            if img.ndim == 2: img = np.expand_dims(img, axis=0)
            elif img.ndim == 3 and img.shape[0] != 1: img = img[0:1, :, :]
            return torch.from_numpy(img)

        hr_tensor = robust_load(self.hr_dir, fname)
        lr_tensor = robust_load(self.lr_dir, fname)
        
        if self.transform:
            seed = np.random.randint(2147483647)
            torch.manual_seed(seed); hr_tensor = self.transform(hr_tensor)
            torch.manual_seed(seed); lr_tensor = self.transform(lr_tensor)
            
        return lr_tensor, hr_tensor

lr_path_ix = '/kaggle/input/datasets/dundikuladeepeswar/dataset9b/Dataset/LR'
hr_path_ix = '/kaggle/input/datasets/dundikuladeepeswar/dataset9b/Dataset/HR'

ix_ds = LensingSR_IX_Dataset(lr_path_ix, hr_path_ix, transform=transforms.RandomHorizontalFlip())
ix_loader = DataLoader(ix_ds, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)

In [6]:
from timm.layers import DropPath, to_2tuple, trunc_normal_

class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)
    def forward(self, x):
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))

class SwinIR(nn.Module):
    def __init__(self, in_chans=1, embed_dim=96, upscale=2):
        super(SwinIR, self).__init__()
        self.conv_first = nn.Conv2d(in_chans, embed_dim, 3, 1, 1)
        self.upsample = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim * (upscale ** 2), 3, 1, 1),
            nn.PixelShuffle(upscale),
            nn.Conv2d(embed_dim, in_chans, 3, 1, 1)
        )
    def forward(self, x):
        return self.upsample(self.conv_first(x))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_ix = SwinIR(upscale=2).to(device)

# Load weights from Task VI.A 
model_ix.load_state_dict(torch.load('/kaggle/input/models/dundikuladeepeswar/deeplense-super-resolution-model/pytorch/default/1/lensing_sr_final.pth', weights_only=True))

if torch.cuda.device_count() > 1:
    model_ix = nn.DataParallel(model_ix)

optimizer = optim.AdamW(model_ix.parameters(), lr=1e-5, weight_decay=1e-4)
criterion = nn.L1Loss()
scaler = torch.amp.GradScaler('cuda')

In [8]:
for epoch in range(10):
    model_ix.train()
    epoch_loss = 0
    for lr, hr in ix_loader:
        lr, hr = lr.to(device), hr.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            sr = model_ix(lr)
            if sr.shape != hr.shape:
                sr = F.interpolate(sr, size=(hr.shape[2], hr.shape[3]), mode='bilinear')
            loss = criterion(sr, hr)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}/10 | Fine-tune Loss: {epoch_loss/len(ix_loader):.6f}")

torch.save(model_ix.state_dict(), 'foundation_sr_finetuned.pth')

Epoch 1/10 | Fine-tune Loss: 0.004971
Epoch 2/10 | Fine-tune Loss: 0.004958
Epoch 3/10 | Fine-tune Loss: 0.004958
Epoch 4/10 | Fine-tune Loss: 0.004958
Epoch 5/10 | Fine-tune Loss: 0.004958
Epoch 6/10 | Fine-tune Loss: 0.004958
Epoch 7/10 | Fine-tune Loss: 0.004958
Epoch 8/10 | Fine-tune Loss: 0.004958
Epoch 9/10 | Fine-tune Loss: 0.004957
Epoch 10/10 | Fine-tune Loss: 0.004957


In [9]:
def evaluate_ix_metrics(model, loader):
    model.eval()
    mse_sum, psnr_sum, ssim_sum = 0, 0, 0
    with torch.no_grad():
        for lr, hr in loader:
            lr, hr = lr.to(device), hr.to(device)
            sr = model(lr).clamp(0, 1)
            if sr.shape != hr.shape:
                sr = F.interpolate(sr, size=(hr.shape[2], hr.shape[3]), mode='bilinear')
            
            mse = F.mse_loss(sr, hr).item()
            mse_sum += mse
            psnr_sum += 10 * torch.log10(1 / (torch.tensor(mse) + 1e-10)).item()
            
            sr_img = sr[0].cpu().numpy().squeeze()
            hr_img = hr[0].cpu().numpy().squeeze()
            ssim_sum += ssim(sr_img, hr_img, data_range=1)
            
    n = len(loader)
    print(f"IX.B Metrics -> MSE: {mse_sum/n:.6f}, PSNR: {psnr_sum/n:.2f}dB, SSIM: {ssim_sum/n:.4f}")

evaluate_ix_metrics(model_ix, ix_loader)

IX.B Metrics -> MSE: 0.000061, PSNR: 42.13dB, SSIM: 0.9771
